# Full channel DNS: streamwise fluctuations in x–z

Preview: ten exported snapshot indices, **400 through 4000 inclusive**, spaced every 400. The lower-wall planes are selected nearest to $y=0.075, 0.30, 1.00$ (near wall, farther from the wall, and centerline). Plot titles use the actual grid coordinates.

At each selected y, subtract a **single time-and-plane mean** from every frame: $u'(t,x,y,z) = u(t,x,y,z) - \langle u \rangle_{t,x,z}(y)$. In preview mode this mean is estimated from the ten snapshots; it is not a full-run mean or a separate mean per frame.

Run settings below. `off` makes no network requests; `synthetic` tests the plot locally; `preview` reads the ten snapshots from A1. `full` is a separate large read: it streams all snapshots to compute the mean, retains only the ten displayed frames, and includes the startup transient. Do not enable live reads until their scope is approved. The full Zarr is opened lazily and remains read-only. Its chunks are whole snapshots, so even a three-plane selection fetches one whole compressed chunk per selected time. Verified preview payload: 358,152,620 bytes (341.6 MiB), once through the ten chunks.

**Saved output:** live A1 DNS from `preview` mode, executed through Jupyter MCP. The saved setting is reset to `off` so rerunning does not refetch data.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

MODE = 'off'  # 'off', 'synthetic', 'preview', or 'full'
BUCKET = 'simulations'
SOURCE = 'flow_export/A1'
ENDPOINT = 'http://localhost:8333'
PLOT_INDICES = tuple(range(400, 4001, 400))
Y_TARGETS = (0.075, 0.30, 1.00)
U_FIELD = 0  # Export layout: (snapshot, field, x, y, z); field 0 is u.


In [2]:
def open_channel():
    import s3fs
    import zarr
    fs = s3fs.S3FileSystem(profile='default', client_kwargs={'endpoint_url': ENDPOINT})
    root = f'{BUCKET}/{SOURCE}'

    def array(path):
        key = f'{root}/{path}'
        fsspec_store = getattr(zarr.storage, 'FsspecStore', None)
        if fsspec_store is not None:
            store = fsspec_store.from_url(
                f's3://{key}', read_only=True,
                storage_options={'profile': 'default', 'client_kwargs': {'endpoint_url': ENDPOINT}},
            )
        else:
            store = fs.get_mapper(key)
        return zarr.open_array(store=store, mode='r')

    ds = array('data.zarr')
    axes = tuple(array(f'channel.zarr/{axis}')[:] for axis in 'xyz')
    return ds, axes


def choose_planes(y, targets):
    if any(not y[0] <= target <= y[-1] for target in targets):
        raise ValueError('A requested y falls outside the grid.')
    iy = tuple(int(np.abs(y - target).argmin()) for target in targets)
    if len(iy) != len(set(iy)):
        raise ValueError('Choose y targets that snap to distinct grid planes.')
    return iy


def stream_fluctuations(shape, axes, iy, mean_indices, plot_indices, read_planes):
    nx, _, nz = shape
    if not len(mean_indices) or min(mean_indices) < 0:
        raise ValueError('Invalid mean selection.')
    keep = set(plot_indices)
    if len(keep) != len(plot_indices):
        raise ValueError('Plot indices must be unique.')
    sums = np.zeros(len(iy), dtype=np.float64)
    frames = {}
    expected = (nx, len(iy), nz)
    count = 0
    for t in mean_indices:
        slab = np.asarray(read_planes(t, iy))
        if slab.shape != expected or not np.isfinite(slab).all():
            raise ValueError(f'Invalid plane data for snapshot {t}: {slab.shape}')
        sums += slab.sum(axis=(0, 2), dtype=np.float64)
        count += 1
        if t in keep:
            frames[t] = slab.copy()
    if set(frames) != keep or count != len(mean_indices):
        raise ValueError('The stream did not return the complete requested selection.')
    mean_u = sums / (count * nx * nz)
    u_prime = np.stack([frames[t] for t in plot_indices]) - mean_u[None, None, :, None]
    return mean_u, u_prime


In [3]:
def plot_xz(x, y, z, iy, plot_indices, u_prime, label):
    # Images are (z, x), whereas each source plane is (x, z).
    limit = float(np.abs(u_prime).max()) or 1.0
    fig, panels = plt.subplots(len(plot_indices), len(iy), sharex=True, sharey=True,
                               squeeze=False, figsize=(11.3, 1.28 * len(plot_indices) + 0.9),
                               layout='constrained')
    for row, t in enumerate(plot_indices):
        for col, index_y in enumerate(iy):
            ax = panels[row, col]
            mesh = ax.pcolormesh(x, z, u_prime[row, :, col, :].T,
                                 shading='nearest', cmap='RdBu_r', vmin=-limit, vmax=limit,
                                 rasterized=True)
            if row == 0:
                ax.set_title(f'y = {y[index_y]:.4f} (grid {index_y})')
            if col == 0:
                ax.set_ylabel(f'{t}  |  z')
            if row == len(plot_indices) - 1:
                ax.set_xlabel('x')
    fig.suptitle(label + '\nRows: exported snapshot index')
    fig.colorbar(mesh, ax=panels.ravel().tolist(), shrink=0.85, label='u′ (export units)')
    return fig


def synthetic_channel():
    x, y, z = np.linspace(0, 8, 32), np.linspace(0, 2, 41), np.linspace(0, 3, 17)
    def read(t, iy):
        return (y[list(iy)][None, :, None]
                + np.sin(x[:, None, None]) * np.cos(z[None, None, :] + t/350))
    return (x, y, z), read


In [4]:
fig = None
if MODE == 'off':
    print('Live reads disabled. Set MODE to synthetic or an approved live mode.')
elif MODE in ('synthetic', 'preview', 'full'):
    if MODE == 'synthetic':
        axes, read_planes = synthetic_channel()
        mean_indices = PLOT_INDICES
        label = 'Synthetic check (not DNS)'
    else:
        ds, axes = open_channel()
        if ds.ndim != 5 or ds.shape[1] <= U_FIELD or PLOT_INDICES[-1] >= ds.shape[0]:
            raise ValueError(f'Unexpected export shape: {ds.shape}')
        if ds.shape[2:] != tuple(map(len, axes)):
            raise ValueError('Data and coordinate shapes differ.')
        mean_indices = PLOT_INDICES if MODE == 'preview' else range(ds.shape[0])
        def read_planes(t, iy):
            # One selection for all requested y planes: do not refetch for each y.
            return ds.oindex[t, U_FIELD, :, list(iy), :]
        label = f'A1 channel DNS: u′ about the {len(mean_indices):,}-snapshot mean'
    x, y, z = axes
    for a in axes:
        if a.ndim != 1 or not np.isfinite(a).all() or not np.all(np.diff(a) > 0):
            raise ValueError('Expected finite increasing coordinate arrays.')
    iy = choose_planes(y, Y_TARGETS)
    mean_u, u_prime = stream_fluctuations(tuple(map(len, axes)), axes, iy, mean_indices,
                                         PLOT_INDICES, read_planes)
    if MODE != 'full':
        np.testing.assert_allclose(u_prime.mean(axis=(0, 1, 3)), 0, atol=1e-12)
    print('Plot indices:', PLOT_INDICES)
    print('Grid y:', y[list(iy)])
    print('Time/plane mean u:', mean_u)
    fig = plot_xz(x, y, z, iy, PLOT_INDICES, u_prime, label)
    plt.show()
    plt.close(fig)
else:
    raise ValueError(f'Unknown MODE: {MODE}')


Live reads disabled. Set MODE to synthetic or an approved live mode.


## 3D low-speed isosurfaces

The saved one-snapshot preview shows the boundary of $u' < -1\,u'_\mathrm{rms}$ at exported snapshot **2000**, near the lower wall ($y\lesssim0.4$). Here $u' = u - \langle u \rangle_{x,z}(t,y)$, subtracting the **same snapshot's** plane mean at every y. This differs from the ten-snapshot mean used above. `ISO_MODE = 'preview_mean'` uses that same ten-snapshot reference, extended to all displayed y planes; it fetches ten entire compressed chunks (341.6 MiB, one pass), so only enable it after approving that read. `ISO_MODE = 'snapshot'` fetches one chunk (34.2 MiB for snapshot 2000); `off` fetches nothing.

The default threshold uses the volume-weighted RMS inside the displayed slab, integrating on the actual y grid. Set `ISO_Y_MAX = None` for the full gap, or set `ISO_LEVEL` to a negative value in export velocity units for a fixed threshold. The surface intersects the actual, nonuniform wall-normal grid and is not stitched across periodic boundaries. Both views exaggerate wall-normal y by 4× for visibility; tick labels remain in export coordinates. This section uses PyVista for contour extraction and Matplotlib for rendering.

In [5]:
ISO_MODE = 'off'  # 'off', 'snapshot', or 'preview_mean'
ISO_SNAPSHOT = 2000
ISO_Y_MAX = 0.4     # None for the full channel
ISO_LEVEL = None    # None for -1 × slab RMS, or a fixed negative u′ value
ISO_Y_EXAGGERATION = 4.0

In [6]:
def iso_velocity(ds, axes, snapshot, mean_indices, y_max, u_field=0):
    x, all_y, z = axes
    if ds.ndim != 5 or tuple(ds.shape[2:]) != tuple(map(len, axes)):
        raise ValueError('Unexpected channel shape.')
    if any(a.ndim != 1 or not np.isfinite(a).all() or not np.all(np.diff(a) > 0) for a in axes):
        raise ValueError('Expected finite increasing coordinates.')
    end = len(all_y) if y_max is None else int(np.searchsorted(all_y, y_max, side='right'))
    if end < 2 or not 0 <= snapshot < ds.shape[0]:
        raise ValueError('Invalid y crop or snapshot.')
    if not mean_indices or len(set(mean_indices)) != len(mean_indices) or any(not 0 <= t < ds.shape[0] for t in mean_indices):
        raise ValueError('Invalid mean indices.')
    means = np.zeros(end, dtype='float64')
    frame = None
    # One chunk per snapshot, including the displayed snapshot if it belongs to the mean.
    for t in dict.fromkeys((*mean_indices, snapshot)):
        u = np.asarray(ds[t, u_field, :, :end, :])
        if u.shape != (len(x), end, len(z)) or not np.isfinite(u).all():
            raise ValueError(f'Invalid velocity data at snapshot {t}.')
        if t in mean_indices:
            means += u.mean(axis=(0, 2), dtype='float64')
        if t == snapshot:
            frame = u
    return (x, all_y[:end], z), (frame - (means / len(mean_indices))[None, :, None]).astype('float32')


def low_speed_surface(axes, u_prime, level=None):
    import pyvista as pv
    x, y, z = axes
    variance = np.square(u_prime, dtype='float64').mean(axis=(0, 2))
    rms = float(np.sqrt(np.sum((variance[:-1] + variance[1:]) * 0.5 * np.diff(y)) / (y[-1] - y[0])))
    level = -rms if level is None else float(level)
    if not (np.isfinite(level) and level < 0 and float(u_prime.min()) < level < float(u_prime.max())):
        raise ValueError(f'No negative contour at {level:g}; field range {u_prime.min():g} to {u_prime.max():g}.')
    grid = pv.RectilinearGrid(x, y, z)
    grid.point_data['u_prime'] = u_prime.ravel(order='F')
    mesh = grid.contour([level], scalars='u_prime', preference='point').triangulate()
    if not mesh.n_points or not mesh.n_cells:
        raise ValueError('Empty isosurface.')
    faces = np.asarray(mesh.faces).reshape(-1, 4)
    assert np.all(faces[:, 0] == 3)
    points = np.asarray(mesh.points)
    assert np.isfinite(points).all()
    return points, faces[:, 1:], level, rms


def plot_low_speed_3d(axes, points, faces, snapshot, level, rms, mean_label, y_exaggeration):
    from matplotlib.colors import LightSource
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    x, y, z = axes
    xyz_display = points[:, [0, 2, 1]]  # x and z along the floor; y vertical
    fig = plt.figure(figsize=(14, 6.4), facecolor='white')
    fig.subplots_adjust(left=0.015, right=0.98, top=0.86, bottom=0.12, wspace=0.02)
    for i, (elev, azim, label) in enumerate(((26, -67, 'Oblique view'), (58, -96, 'View from above')), 1):
        ax = fig.add_subplot(1, 2, i, projection='3d', proj_type='ortho')
        surface = Poly3DCollection(xyz_display[faces], facecolors='#2474ad',
                                   shade=True, lightsource=LightSource(azdeg=315, altdeg=55))
        surface.set_edgecolor('none')
        ax.add_collection3d(surface)
        ax.set(xlim=(x[0], x[-1]), ylim=(z[0], z[-1]), zlim=(y[0], y[-1]),
               xlabel='x', ylabel='z', zlabel='y', title=label)
        ax.set_box_aspect((np.ptp(x), np.ptp(z), np.ptp(y)*y_exaggeration), zoom=0.97)
        ax.set_xticks(np.linspace(x[0], x[-1], 5).round(1))
        ax.set_yticks(np.linspace(z[0], z[-1], 3).round(1))
        ax.set_zticks([y[0], y[-1]])
        ax.set_zticklabels([f'{y[0]:.2f}', f'{y[-1]:.2f}'])
        ax.tick_params(labelsize=8, pad=0)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.labelpad = 7
        ax.view_init(elev=elev, azim=azim)
    fig.suptitle(f'Low-speed isosurfaces | A1 snapshot {snapshot}', fontsize=17, fontweight='semibold', y=0.95)
    fig.text(0.5, 0.075, f"u′ = {level:.4f}   |   slab RMS = {rms:.4f}   |   {mean_label}", ha='center', fontsize=11)
    fig.text(0.5, 0.035, f'{y[0]:.3f} ≤ y ≤ {y[-1]:.3f}   |   Wall-normal scale exaggerated {y_exaggeration:g}×', ha='center', fontsize=10)
    return fig


In [7]:
iso_fig = None
if ISO_MODE == 'off':
    print('3D live reads disabled. Select an approved ISO_MODE to regenerate.')
elif ISO_MODE in ('snapshot', 'preview_mean'):
    iso_ds, iso_axes = open_channel()
    iso_mean_indices = (ISO_SNAPSHOT,) if ISO_MODE == 'snapshot' else PLOT_INDICES
    slab_axes, iso_up = iso_velocity(iso_ds, iso_axes, ISO_SNAPSHOT, iso_mean_indices, ISO_Y_MAX, U_FIELD)
    if ISO_MODE == 'snapshot':
        np.testing.assert_allclose(iso_up.mean(axis=(0, 2), dtype='float64'), 0.0, atol=1e-8)
    iso_points, iso_faces, iso_level, iso_rms = low_speed_surface(slab_axes, iso_up, ISO_LEVEL)
    iso_label = 'Snapshot plane mean' if ISO_MODE == 'snapshot' else f'{len(iso_mean_indices)}-snapshot time/plane mean'
    print(f'Snapshot {ISO_SNAPSHOT}; y = [{slab_axes[1][0]:.5f}, {slab_axes[1][-1]:.5f}]; '
          f'level {iso_level:.6f}; slab RMS {iso_rms:.6f}; {len(iso_faces):,} triangles; '
          f'mean indices {iso_mean_indices}')
    iso_fig = plot_low_speed_3d(slab_axes, iso_points, iso_faces, ISO_SNAPSHOT, iso_level,
                                iso_rms, iso_label, ISO_Y_EXAGGERATION)
    plt.show()
    plt.close(iso_fig)
else:
    raise ValueError(f'Unknown ISO_MODE: {ISO_MODE}')


3D live reads disabled. Select an approved ISO_MODE to regenerate.


## Q-criterion isosurfaces

Uses `critq` (zero-based field 4) from the A1 export, with no mean subtraction or recomputation. The channel solver writes $Q=-\tfrac12\mathrm{tr}[(\nabla\mathbf u)^2]$; positive Q identifies regions where local rotation dominates strain. Snapshot **2000** and the lower-wall slab ($y\leq 0.4$) match the low-speed 3D preview for a visual comparison.

By default the isovalue is the 95th percentile of **positive sampled Q** in the displayed slab. This is a visualization threshold, not a universal vortex cutoff; it is a grid-point percentile, not a volume-weighted percentile. Set `Q_LEVEL` to a fixed positive Q in export units for comparisons across snapshots. Set `Q_Y_MAX = None` for the full gap. Wall-normal scale is exaggerated 4×, and the mesh is not stitched across periodic boundaries. The saved output is live DNS; `Q_ENABLED = False` makes reruns read no data until explicitly enabled.

In [8]:
# Configure the Q-criterion preview; live reads are opt-in.
Q_ENABLED = False
Q_SNAPSHOT = 2000
Q_FIELD = 4
Q_Y_MAX = 0.4
Q_LEVEL = None
Q_POSITIVE_PERCENTILE = 95.0
Q_Y_EXAGGERATION = 4.0

In [9]:
# Extract positive Q contours on the exported nonuniform grid and show two views.
def q_isosurface(axes, field, level=None, percentile=95.0):
    import pyvista as pv
    x, y, z = axes
    if np.shape(field) != (len(x), len(y), len(z)) or not np.isfinite(field).all():
        raise ValueError('Invalid Q field or shape.')
    if any(a.ndim != 1 or len(a) < 2 or not np.isfinite(a).all() or not np.all(np.diff(a) > 0) for a in axes):
        raise ValueError('Expected finite increasing grid coordinates.')
    if level is None:
        positive = field[field > 0]
        if not positive.size or not 0 < percentile < 100:
            raise ValueError('Positive Q and a percentile between 0 and 100 are required.')
        level = float(np.percentile(positive, percentile))
    if not (np.isfinite(level) and max(0.0, float(field.min())) < level < float(field.max())):
        raise ValueError(f'No valid positive contour at {level:g} for this slab.')
    grid = pv.RectilinearGrid(x, y, z)
    grid.point_data['Q'] = field.ravel(order='F')
    mesh = grid.contour([level], scalars='Q', preference='point').triangulate()
    if not mesh.n_points or not mesh.n_cells:
        raise ValueError('Empty Q isosurface.')
    faces = np.asarray(mesh.faces).reshape(-1, 4)
    points = np.asarray(mesh.points)
    assert np.isfinite(points).all() and np.all(faces[:, 0] == 3)
    return points, faces[:, 1:], level


def plot_q_views(axes, points, faces, snapshot, level, threshold_label, y_exaggeration):
    from matplotlib.colors import LightSource
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    x, y, z = axes
    triangles = points[:, [0, 2, 1]][faces]
    fig = plt.figure(figsize=(14, 6.4), facecolor='white')
    fig.subplots_adjust(left=0.015, right=0.98, top=0.86, bottom=0.12, wspace=0.02)
    for i, (elev, azim, label) in enumerate(((26, -67, 'Oblique view'), (58, -96, 'View from above')), 1):
        ax = fig.add_subplot(1, 2, i, projection='3d', proj_type='ortho')
        mesh = Poly3DCollection(triangles, facecolors='#e58631', shade=True,
                                lightsource=LightSource(azdeg=315, altdeg=55))
        mesh.set_edgecolor('none')
        ax.add_collection3d(mesh)
        ax.set(xlim=(x[0], x[-1]), ylim=(z[0], z[-1]), zlim=(y[0], y[-1]),
               xlabel='x', ylabel='z', zlabel='y', title=label)
        ax.set_box_aspect((np.ptp(x), np.ptp(z), np.ptp(y)*y_exaggeration), zoom=0.97)
        ax.set_xticks(np.linspace(x[0], x[-1], 5).round(1))
        ax.set_yticks(np.linspace(z[0], z[-1], 3).round(1))
        ax.set_zticks([y[0], y[-1]])
        ax.set_zticklabels([f'{y[0]:.2f}', f'{y[-1]:.2f}'])
        ax.tick_params(labelsize=8, pad=0)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.labelpad = 7
        ax.view_init(elev=elev, azim=azim)
    fig.suptitle(f'Q-criterion isosurfaces | A1 snapshot {snapshot}', fontsize=17, fontweight='semibold', y=0.95)
    fig.text(0.5, 0.075, f'Q = {level:.4g} (export units)   |   {threshold_label}', ha='center', fontsize=11)
    fig.text(0.5, 0.035, f'{y[0]:.3f} ≤ y ≤ {y[-1]:.3f}   |   Wall-normal scale exaggerated {y_exaggeration:g}×', ha='center', fontsize=10)
    return fig

In [10]:
# Read one exported Q slab and render the surface only when enabled.
q_fig = None
if Q_ENABLED:
    q_ds, q_axes_all = open_channel()
    if q_ds.ndim != 5 or q_ds.shape[2:] != tuple(map(len, q_axes_all)) or not 0 <= Q_FIELD < q_ds.shape[1] or not 0 <= Q_SNAPSHOT < q_ds.shape[0]:
        raise ValueError('Unexpected channel shape, field or snapshot selection.')
    qx, qy, qz = q_axes_all
    q_end = len(qy) if Q_Y_MAX is None else int(np.searchsorted(qy, Q_Y_MAX, side='right'))
    if q_end < 2:
        raise ValueError('The Q slab must contain at least two y coordinates.')
    q_axes = qx, qy[:q_end], qz
    q_field = q_ds[Q_SNAPSHOT, Q_FIELD, :, :q_end, :]
    q_points, q_faces, q_level = q_isosurface(q_axes, q_field, Q_LEVEL, Q_POSITIVE_PERCENTILE)
    q_label = f'{Q_POSITIVE_PERCENTILE:g}th percentile of positive Q' if Q_LEVEL is None else 'fixed positive threshold'
    print(f'Snapshot {Q_SNAPSHOT}; y=[{qy[0]:.5f}, {qy[q_end-1]:.5f}]; Q={q_level:.7g}; {len(q_faces):,} triangles')
    q_fig = plot_q_views(q_axes, q_points, q_faces, Q_SNAPSHOT, q_level, q_label, Q_Y_EXAGGERATION)
    plt.show()
    plt.close(q_fig)
else:
    print('Q live reads disabled. Set Q_ENABLED = True to regenerate this preview.')

Q live reads disabled. Set Q_ENABLED = True to regenerate this preview.


## Reynolds shear-stress structures: $u'v'$

The three panels show intense **instantaneous** $u'v'$ structures from the same snapshot and lower-wall slab used above. Both velocity components are centered by their **own x–z means at each y in snapshot 2000**. No time averaging is applied to the product. Here the exported wall-normal velocity is positive away from the lower wall.

- **Quadrant 2 / ejections:** $u'<0$, $v'>0$, so $u'v'<0$.
- **Quadrant 4 / sweeps:** $u'>0$, $v'<0$, so $u'v'<0$.
- **Quadrants 1 and 3:** same-sign fluctuations, $u'v'>0$.

All panels use the same magnitude threshold: the 97.5th percentile of $|u'v'|$ over the sampled grid points in the slab (not volume-weighted), or a fixed positive `UV_LEVEL` in squared export velocity units. Each quadrant field is masked to zero outside its own sign combination before contour extraction. The y axis is exaggerated 4×, and periodic boundaries are not stitched. `UV_ENABLED = False` prevents live reads by default. Enabling this preview fetches one whole snapshot chunk for both velocity components together.

In [11]:
# Configure the shear-stress preview; live reads are opt-in.
UV_ENABLED = False
UV_SNAPSHOT = 2000
UV_Y_MAX = 0.4  # None for the full gap; then Q2/Q4 remain referenced to +v at the lower wall.
UV_PERCENTILE = 97.5
UV_LEVEL = None
UV_Y_EXAGGERATION = 4.0

In [12]:
# Center u and v on each plane, then isolate the three signed product fields.
def uv_quadrants(velocity):
    if velocity.ndim != 4 or velocity.shape[0] != 2 or not np.isfinite(velocity).all():
        raise ValueError('Expected two finite fields in (component, x, y, z) order.')
    u, v = velocity - velocity.mean(axis=(1, 3), dtype='float64')[:, None, :, None]
    uv = u*v
    fields = (np.where((u < 0) & (v > 0), -uv, 0.0),
              np.where((u > 0) & (v < 0), -uv, 0.0),
              np.maximum(uv, 0.0))
    return uv, fields


def plot_uv_structures(axes, fields, snapshot, level, level_label, exaggeration):
    from matplotlib.colors import LightSource
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    x, y, z = axes
    panels = [('Ejections (quadrant 2)', 'u′ < 0, v′ > 0', '#237ebe'),
              ('Sweeps (quadrant 4)', 'u′ > 0, v′ < 0', '#d67527'),
              ('Same-sign events (quadrants 1 & 3)', 'u′v′ > 0', '#8155a6')]
    fig = plt.figure(figsize=(16, 5.55), facecolor='white')
    fig.subplots_adjust(left=0.012, right=0.985, top=0.83, bottom=0.16, wspace=0.075)
    counts = []
    for i, (field, (name, subtitle, color)) in enumerate(zip(fields, panels), 1):
        ax = fig.add_subplot(1, 3, i, projection='3d', proj_type='ortho')
        if field.min() < level < field.max():
            points, faces, _ = q_isosurface(axes, field, level)
            obj = Poly3DCollection(points[:, [0, 2, 1]][faces], facecolors=color,
                                    shade=True, lightsource=LightSource(azdeg=320, altdeg=55))
            obj.set_edgecolor('none')
            ax.add_collection3d(obj)
            counts.append(len(faces))
        else:
            ax.text2D(0.5, 0.48, 'No contour at this threshold', ha='center', transform=ax.transAxes)
            counts.append(0)
        ax.set(xlim=(x[0], x[-1]), ylim=(z[0], z[-1]), zlim=(y[0], y[-1]), xlabel='x', ylabel='z', zlabel='y')
        ax.set_title(f'{name}\n{subtitle}', color=color, fontsize=10.5, pad=4)
        ax.set_box_aspect((np.ptp(x), np.ptp(z), np.ptp(y)*exaggeration), zoom=0.94)
        ax.set_xticks(np.linspace(x[0], x[-1], 5).round(1))
        ax.set_yticks(np.linspace(z[0], z[-1], 3).round(1))
        ax.set_zticks([y[0], y[-1]])
        ax.set_zticklabels([f'{y[0]:.2f}', f'{y[-1]:.2f}'])
        ax.tick_params(labelsize=8, pad=0)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.labelpad = 5
        ax.view_init(elev=48, azim=-78)
    fig.suptitle(f'u′v′ isosurfaces | A1 snapshot {snapshot}', fontsize=17, fontweight='semibold', y=0.95)
    fig.text(0.5, 0.10, f'|u′v′| = {level:.5g}   |   {level_label}', ha='center', fontsize=11)
    fig.text(0.5, 0.055, f'{y[0]:.3f} ≤ y ≤ {y[-1]:.3f}   |   Snapshot x–z means removed   |   y exaggerated {exaggeration:g}×', ha='center', fontsize=10)
    return fig, counts

In [13]:
# Fetch u and v in one selection and plot the signed structures.
uv_fig = None
if UV_ENABLED:
    uv_ds, uv_axes_all = open_channel()
    if uv_ds.ndim != 5 or uv_ds.shape[1] < 2 or uv_ds.shape[2:] != tuple(map(len, uv_axes_all)) or not 0 <= UV_SNAPSHOT < uv_ds.shape[0]:
        raise ValueError('Invalid channel shape or snapshot index.')
    ux, uy, uz = uv_axes_all
    uv_end = len(uy) if UV_Y_MAX is None else int(np.searchsorted(uy, UV_Y_MAX, side='right'))
    if uv_end < 2:
        raise ValueError('The slab must contain at least two y coordinates.')
    uv_axes = ux, uy[:uv_end], uz
    uv_velocity = np.asarray(uv_ds[UV_SNAPSHOT, :2, :, :uv_end, :])  # fields 0 and 1: u and v
    uv_product, uv_fields = uv_quadrants(uv_velocity)
    if UV_LEVEL is None and not 0 < UV_PERCENTILE < 100:
        raise ValueError('Set a percentile between 0 and 100.')
    uv_level = float(np.percentile(np.abs(uv_product), UV_PERCENTILE)) if UV_LEVEL is None else float(UV_LEVEL)
    if not np.isfinite(uv_level) or uv_level <= 0:
        raise ValueError('Set a finite positive magnitude threshold.')
    uv_label = f'{UV_PERCENTILE:g}th percentile of the slab magnitude (grid points)' if UV_LEVEL is None else 'fixed threshold in squared export velocity units'
    uv_fig, uv_counts = plot_uv_structures(uv_axes, uv_fields, UV_SNAPSHOT, uv_level, uv_label, UV_Y_EXAGGERATION)
    print(f'Snapshot {UV_SNAPSHOT}; y=[{uy[0]:.5f}, {uy[uv_end-1]:.5f}]; |u′v′|={uv_level:.8g}; triangles {uv_counts} (Q2, Q4, Q1+Q3)')
    plt.show()
    plt.close(uv_fig)
else:
    print('Live u′v′ reads disabled. Set UV_ENABLED = True to regenerate this preview.')

Live u′v′ reads disabled. Set UV_ENABLED = True to regenerate this preview.


## Interactive PyVista isosurfaces

Run the setup cells above, then set `INTERACTIVE_ENABLED = True` below. Choose `low_speed`, `q`, or `uv` (ejections, sweeps, and same-sign events in one view). The slider starts at the static plot's threshold magnitude and updates only when released. Rotate and zoom with the mouse.

The first launch reads one snapshot chunk and caches its slab in this kernel. Switching fields on the same slab reuses that cache. Moving the slider never reads S3. Defaults match the static plots: snapshot 2000, lower-wall slab through `y <= 0.4`, and 4× wall-normal exaggeration. Set `INTERACTIVE_Y_MAX = None` for the full gap.

The live display uses PyVista's Trame server backend. For a remote Jupyter server, keep `INTERACTIVE_PROXY = True` and enable Jupyter Server Proxy. For local Jupyter with direct Trame access, set it to `False`.

In [14]:
# Load at most one slab and update PyVista contours without reloading any fields.
def interactive_slab(snapshot, y_max):
    cache = globals().setdefault('_interactive_slab_cache', {})
    key = (BUCKET, SOURCE, ENDPOINT, snapshot, y_max)
    if key not in cache:
        ds, (x, y, z) = open_channel()
        if ds.ndim != 5 or ds.shape[1] < 5 or ds.shape[2:] != (len(x), len(y), len(z)) or not 0 <= snapshot < ds.shape[0]:
            raise ValueError('Invalid export shape or snapshot index.')
        n = len(y) if y_max is None else int(np.searchsorted(y, y_max, side='right'))
        if n < 2:
            raise ValueError('The slab needs at least two y planes.')
        raw = ds[snapshot, :5, :, :n, :]  # All needed fields share one chunk.
        if not np.isfinite(raw).all():
            raise ValueError('The slab contains nonfinite values.')
        cache.clear()
        cache[key] = ((x, y[:n], z), raw[:2].copy(), raw[4].copy())
    return cache[key]


def interactive_iso_view(axes, velocity, q, field, snapshot, proxy=True):
    import pyvista as pv
    x, y, z = axes
    if any(len(a) < 2 or not np.isfinite(a).all() or np.any(np.diff(a) <= 0) for a in axes):
        raise ValueError('Expected finite increasing coordinates.')
    if field == 'low_speed':
        u = velocity[0].astype('float64')
        u -= u.mean(axis=(0, 2))[None, :, None]
        m = np.square(u).mean(axis=(0, 2))
        base = np.sqrt(np.sum(0.5 * (m[:-1] + m[1:]) * np.diff(y)) / np.ptp(y))
        specs, label, sign = [('Low speed', '#2474ad', u)], 'u′', -1
    elif field == 'q':
        if not np.any(q > 0):
            raise ValueError('No positive Q in this slab.')
        base = np.percentile(q[q > 0], 95)
        specs, label, sign = [('Q', '#e58631', q)], 'Q', 1
    elif field == 'uv':
        uv, groups = uv_quadrants(velocity)
        base = np.percentile(np.abs(uv), 97.5)
        specs = list(zip(('Ejections', 'Sweeps', 'Same-sign'), ('#237ebe', '#d67527', '#8155a6'), groups))
        label, sign = '|u′v′|', 1
    else:
        raise ValueError("Choose 'low_speed', 'q', or 'uv'.")
    base = float(base)
    if not np.isfinite(base) or base <= 0:
        raise ValueError('No finite, positive threshold magnitude.')

    grid = pv.RectilinearGrid(x, z, y)  # Make wall-normal y vertical.
    bounds = {}
    for name, color, values in specs:
        if values.shape != (len(x), len(y), len(z)) or not np.isfinite(values).all():
            raise ValueError(f'Invalid scalar field: {name}.')
        grid.point_data[name] = values.transpose(0, 2, 1).ravel(order='F')
        bounds[name] = float(values.min()), float(values.max())
    pv.global_theme.trame.server_proxy_enabled = proxy
    pv.global_theme.trame.jupyter_extension_enabled = False
    pl = pv.Plotter(notebook=True, off_screen=True, window_size=(990, 570))
    pl.set_background('#f5f7fa')
    pl.add_mesh(grid.outline(), color='#64748b', opacity=0.45)
    pl.show_bounds(xtitle='x', ytitle='z', ztitle='y', axes_ranges=grid.bounds,
                   bounds=grid.bounds, n_xlabels=5, n_ylabels=3, n_zlabels=2)
    pl.set_scale(zscale=4.0, reset_camera=False, render=False)
    pl.add_text(f'A1 snapshot {snapshot} | {field} | y exaggerated 4x', position='upper_left', font_size=12)
    caption = pl.add_text('', position=(12, 14), font_size=11)
    state = {}

    def update(magnitude):
        level = float(magnitude) * sign
        counts = {}
        for name, color, _ in specs:
            lo, hi = bounds[name]
            mesh = grid.contour([level], scalars=name) if lo < level < hi else None
            actor = f'iso-{name}'
            if mesh is not None and mesh.n_cells:
                pl.add_mesh(mesh, name=actor, color=color, show_scalar_bar=False, reset_camera=False, render=False)
                counts[name] = mesh.n_cells
            else:
                pl.remove_actor(actor, reset_camera=False, render=False)
                counts[name] = 0
        caption.SetInput(f'{label} = {level:.5g}' + (' | no surface' if not any(counts.values()) else ''))
        state.update(level=level, counts=counts)
        pl.render()

    slider = pl.add_slider_widget(update, (0.6 * base, 2.0 * base), value=base,
                                  title='Threshold magnitude', fmt='%.4g', color='#22314b',
                                  pointa=(0.28, 0.14), pointb=(0.72, 0.14), interaction_event='end')
    if field == 'uv':
        pl.add_legend([(name, color) for name, color, _ in specs], loc='upper right', size=(0.2, 0.15))
    pl.view_isometric()
    return pl, slider, state

In [17]:
# Select a field and open the viewer when ready.
INTERACTIVE_ENABLED = False
INTERACTIVE_FIELD = 'q'  # 'low_speed', 'q', or 'uv'
INTERACTIVE_SNAPSHOT = 2000
INTERACTIVE_Y_MAX = 0.4
INTERACTIVE_PROXY = True

if INTERACTIVE_ENABLED:
    slab = interactive_slab(INTERACTIVE_SNAPSHOT, INTERACTIVE_Y_MAX)
    if globals().get('interactive_plotter') is not None:
        interactive_plotter.close()
    interactive_plotter, interactive_slider, interactive_state = interactive_iso_view(
        *slab, INTERACTIVE_FIELD, INTERACTIVE_SNAPSHOT, proxy=INTERACTIVE_PROXY)
    interactive_plotter.show(jupyter_backend='server', auto_close=False)
else:
    print('Interactive viewer is off. Set INTERACTIVE_ENABLED = True to open it.')

Interactive viewer is off. Set INTERACTIVE_ENABLED = True to open it.
